# 03) Prepare CodeT5 patch dataset from CVEfixes

Builds `vulnerable_code -> fixed_code` pairs from `file_change.code_before` and `file_change.code_after`.
Outputs: `codet5_patch_pairs.csv`

In [ ]:
import os
import sqlite3
import pandas as pd
from pathlib import Path

DATASET_DIR = Path('/kaggle/working')
CVEFIXES_DIR = Path('CVEfixes_v1.0.0')  # adjust as needed
SQL_GZ_PATH = CVEFIXES_DIR / 'Data' / 'CVEfixes-2021-06-09.sql.gz'
SQLITE_PATH = DATASET_DIR / 'CVEfixes.db'
OUT_CSV = DATASET_DIR / 'codet5_patch_pairs.csv'

MAX_PAIRS = int(os.getenv('MAX_PAIRS', '3000'))
DATASET_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
if not SQLITE_PATH.exists():
    if not SQL_GZ_PATH.exists():
        raise FileNotFoundError(f'Missing SQL dump: {SQL_GZ_PATH}')
    !bash -lc "gunzip -c '{SQL_GZ_PATH}' | sqlite3 '{SQLITE_PATH}'"

conn = sqlite3.connect(str(SQLITE_PATH))
sql = f"""
SELECT
  f.code_before AS vulnerable_code,
  f.code_after AS fixed_code,
  cc.cwe_id AS cwe_id,
  cv.cve_id AS cve_id
FROM file_change f
JOIN commits c ON f.hash = c.hash
JOIN fixes fx ON fx.hash = c.hash
JOIN cve cv ON cv.cve_id = fx.cve_id
LEFT JOIN cwe_classification cc ON cc.cve_id = cv.cve_id
WHERE f.programming_language='Python'
  AND f.code_before IS NOT NULL
  AND f.code_after IS NOT NULL
LIMIT {MAX_PAIRS}
"""

df = pd.read_sql_query(sql, conn)
conn.close()

df = df.dropna(subset=['vulnerable_code', 'fixed_code'])
df = df[df['vulnerable_code'].str.len() > 0]
df = df[df['fixed_code'].str.len() > 0]

print(df.head())
print('Rows:', len(df))

df.to_csv(OUT_CSV, index=False)
print('Saved:', OUT_CSV)